In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark = SparkSession.builder.appName("Ecom data pipeline").getOrCreate()

userDf = spark.read.format("delta").load("/mnt/delta/tables/bronze/users")


In [0]:
userDf = userDf.withColumn("countryCode", upper(col("countryCode")))

In [0]:
userDf = userDf.withColumn("language_full",
                           expr("CASE WHEN language = 'EN' THEN 'English' " +
                                "WHEN language = 'FR'THEN 'French' " +
                                "ELSE 'Other' END"))

userDF = userDf.withColumn("gender", expr("CASE WHEN gender = 'M' THEN 'Male' " +
                                            "WHEN gender = 'F' THEN 'Female' " +
                                            "ELSE 'Other' END"))
userDf = userDf.withColumn("civilitytitle_clean",
                           regexp_replace("civilitytitle", "(Mme|Ms|Mrs)","MS"))

userDf = userDf.withColumn("years_since_last_login", col("dayssincelastlogin")/365)

userDf = userDf.withColumn("account_age_years", round(col("seniority")/365,2))

userDf = userDf.withColumn("account_age_group" , 
                           when(col("account_age_years") < 1, "New") 
                           .when((col("account_age_years") >= 1) & (col("account_age_years") < 3), "Intermediate") 
                           .otherwise( "Senior"))



In [0]:
userDf.show(10)

In [0]:
userDf = userDf.withColumn("hasanyapp",col("hasanyapp").cast("boolean"))
userDf = userDf.withColumn("hasandroidapp",col("hasandroidapp").cast("boolean"))
userDf = userDf.withColumn("hasiosapp",col("hasiosapp").cast("boolean"))
userDf = userDf.withColumn("hasprofilepicture",col("hasprofilepicture").cast("boolean"))
userDf = userDf.withColumn("socialnbfollowers",col("socialnbfollowers").cast(IntegerType()))
userDf = userDf.withColumn("socialnbfollows",col("socialnbfollows").cast(IntegerType()))
userDf = userDf.withColumn("productspassrate", col("productspassrate").cast(DecimalType(10,2)))
userDf = userDf.withColumn("seniorityasmonths", col("seniorityasmonths").cast(DecimalType(10,2)))
userDf = userDf.withColumn("seniorityasyears", col("seniorityasyears").cast(DecimalType(10,2)))



In [0]:
userDf = userDf.withColumn("dayssincelastlogin",
                           when(col("dayssincelastlogin").isNotNull(),
                                col("dayssincelastlogin").cast(IntegerType()))
                           .otherwise(0))

In [0]:
userDf = userDf.withColumn("user_descriptor",
                           concat(col("gender"),lit("_")
                                  ,col("countrycode"), lit("_"),
                                  expr("substring(civilitytitle_clean, 1, 3)"), lit("_"),
                                  col("language_full")))

In [0]:
userDf.write.format("delta").mode("overwrite").save("/mnt/delta/tables/silver/users")

In [0]:
BuyerDf = spark.read.format("delta").load("/mnt/delta/tables/bronze/buyers")


In [0]:
BuyerDf.show(5)


In [0]:
integer_columns = ['buyers', 'topbuyers', 'femalebuyers', 'malebuyers', 'topfemalebuyers', 'topmalebuyers', 'totalproductsbought', 'totalproductswished', 'totalproductsliked', 'toptotalproductsbought', 'toptotalproductswished','totalproductsliked']

for column in integer_columns:
    BuyerDf = BuyerDf.withColumn(column, col(column).cast(IntegerType()))

In [0]:
decimal_column = ['topbuyerratio', 'femalebuyersratio', 'topfemalebuyersratio', 'boughtperwishlistratio', 'boughtperlikeratio', 'topboughtperwishlistratio', 'topboughtperlikeratio', 'meanproductsbought', 'meanproductswished', 'meanproductsliked', 'topmeanproductsbought', 'topmeanproductswished', 'topmeanproductsliked', 'meanofflinedays', 'topmeanofflinedays', 'meanfollowers', 'meanfollowing', 'topmeanfollowers', 'topmeanfollowing']

for column in decimal_column:
    BuyerDf = BuyerDf.withColumn(column, col(column).cast(DecimalType(10,2)))


In [0]:
BuyerDf = BuyerDf.withColumn("country", initcap(col("country")))

for column in integer_columns:
    BuyerDf = BuyerDf.fillna({column:0})

BuyerDf = BuyerDf.withColumn("female_to_male_ratio", round(col("femalebuyers")/(col("malebuyers")+1),2))

BuyerDf = BuyerDf.withColumn("wishlist_to_purchase_ratio",
                             round(col("totalproductswished")/(col("totalproductsbought")+1),2))

high_engagement_threshold = 0.5
BuyerDf = BuyerDf.withColumn("high_engagement", 
                             when(col("boughtperwishlistratio") > high_engagement_threshold, True)
                             .otherwise(False))
BuyerDf = BuyerDf.withColumn("growing_female_market",
                             when(col("femalebuyersratio") > col("topfemalebuyersratio"), True)
                             .otherwise(False))



In [0]:
BuyerDf.write.format("delta").mode("overwrite").save("/mnt/delta/tables/silver/buyers")

In [0]:
sellersDf = spark.read.format("delta").load("/mnt/delta/tables/bronze/sellers")


In [0]:
sellersDf = sellersDf \
            .withColumn("nbsellers", col("nbsellers").cast(IntegerType())) \
            .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
            .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
            .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10,2))) \
            .withColumn("meanproductsold", col("meanproductssold").cast(DecimalType(10,2))) \
            .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10,2))) \
            .withColumn("meanfollows", col("meanfollows").cast(DecimalType(10,2))) \
            .withColumn("totalproductsliked", col("totalproductsliked").cast(IntegerType())) \
            .withColumn("totalbought", col("totalbought").cast(IntegerType())) \
            .withColumn("totalwished", col("totalwished").cast(IntegerType())) \
            .withColumn("meanproductsliked", col("meanproductsliked").cast(DecimalType(10,2))) \
            .withColumn("meanproductsbought", col("meanproductsbought").cast(DecimalType(10,2))) \
            .withColumn("meanproductswished", col("meanproductswished").cast(DecimalType(10,2))) \
            .withColumn("percentofappusers", col("percentofappusers").cast(DecimalType(10,2))) \
            .withColumn("percentofiosusers", col("percentofiosusers").cast(DecimalType(10,2))) \
            .withColumn("meanseniority", col("meanseniority").cast(DecimalType(10,2))) 
            

In [0]:
sellersDf = sellersDf.withColumn("country", initcap(col("country"))) \
                     .withColumn("sex",upper(col("sex")))
                                 
sellersDf = sellersDf.withColumn("seller_size_category",
                                 when(col("nbsellers") < 500, "Small")\
                                .when((col("nbsellers") >= 500) & (col("nbsellers") < 2000), "Medium")\
                                .otherwise("Large"))

sellersDf = sellersDf.withColumn("mean_products_listed_per_seller",
                                 round(col("totalproductslisted")/col("nbsellers"),2))

sellersDf = sellersDf.withColumn("mean_products_sold_per_seller",
                                 round(col("totalproductssold")/col("nbsellers"),2))

sellersDf = sellersDf.withColumn("high_seller_pass_rate",
                                 when(col("meansellerpassrate") > 0.75, "High")\
                                     .otherwise("Normal"))

mean_pass_rate = sellersDf.select(round(avg("meansellerpassrate"),2).alias("avg_pass_rate")).collect()[0]["avg_pass_rate"]

sellersDf = sellersDf.withColumn("meansellerpassrate",
                                 when(col("meansellerpassrate").isNull(), mean_pass_rate)
                                 .otherwise(col("meansellerpassrate")))



sellersDf.write.format("delta").mode("overwrite").save("/mnt/delta/tables/silver/sellers")

In [0]:
countriesDF = spark.read.format("delta").load("/mnt/delta/tables/bronze/countries")

countriesDF = countriesDF \
    .withColumn("sellers", col("sellers").cast(IntegerType())) \
    .withColumn("topsellers", col("topsellers").cast(IntegerType())) \
    .withColumn("topsellerratio", col("topsellerratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellersratio", col("femalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("topfemalesellersratio", col("topfemalesellersratio").cast(DecimalType(10, 2))) \
    .withColumn("femalesellers", col("femalesellers").cast(IntegerType())) \
    .withColumn("malesellers", col("malesellers").cast(IntegerType())) \
    .withColumn("topfemalesellers", col("topfemalesellers").cast(IntegerType())) \
    .withColumn("topmalesellers", col("topmalesellers").cast(IntegerType())) \
    .withColumn("countrysoldratio", col("countrysoldratio").cast(DecimalType(10, 2))) \
    .withColumn("bestsoldratio", col("bestsoldratio").cast(DecimalType(10, 2))) \
    .withColumn("toptotalproductssold", col("toptotalproductssold").cast(IntegerType())) \
    .withColumn("totalproductssold", col("totalproductssold").cast(IntegerType())) \
    .withColumn("toptotalproductslisted", col("toptotalproductslisted").cast(IntegerType())) \
    .withColumn("totalproductslisted", col("totalproductslisted").cast(IntegerType())) \
    .withColumn("topmeanproductssold", col("topmeanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("topmeanproductslisted", col("topmeanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanproductssold", col("meanproductssold").cast(DecimalType(10, 2))) \
    .withColumn("meanproductslisted", col("meanproductslisted").cast(DecimalType(10, 2))) \
    .withColumn("meanofflinedays", col("meanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("topmeanofflinedays", col("topmeanofflinedays").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowers", col("meanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("meanfollowing", col("meanfollowing").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowers", col("topmeanfollowers").cast(DecimalType(10, 2))) \
    .withColumn("topmeanfollowing", col("topmeanfollowing").cast(DecimalType(10, 2)))

countriesDF = countriesDF.withColumn("country", initcap(col("country")))


# Calculating the ratio of top sellers to total sellers
countriesDF = countriesDF.withColumn("top_seller_ratio", 
                                        round(col("topsellers") / col("sellers"), 2))

# countriesDF countries with a high ratio of female sellers
countriesDF = countriesDF.withColumn("high_female_seller_ratio", 
                                        when(col("femalesellersratio") > 0.5, True).otherwise(False))

# Adding a performance indicator based on the sold/listed ratio
countriesDF = countriesDF.withColumn("performance_indicator", 
                                        round(col("toptotalproductssold") / (col("toptotalproductslisted") + 1), 2))

# Flag countries with exceptionally high performance
performance_threshold = 0.8
countriesDF = countriesDF.withColumn("high_performance", 
                                        when(col("performance_indicator") > performance_threshold, True).otherwise(False))

countriesDF = countriesDF.withColumn("activity_level",
                                       when(col("meanofflinedays") < 30, "Highly Active")
                                       .when((col("meanofflinedays") >= 30) & (col("meanofflinedays") < 60), "Moderately Active")
                                       .otherwise("Low Activity"))


countriesDF.write.format("delta").mode("overwrite").save("/mnt/delta/tables/silver/countries")